# nearmiss — hotspots (Davis demo)

A reproducible analysis notebook. It loads the city config, runs the full pipeline and the
exposure-normalized statistics over the synthetic Davis fixtures, and renders the ranked
segments and the rate chart. The figures it shows are produced by `nearmiss.figures`, the
same deterministic code that `make reproduce` regenerates and diffs — so this notebook and the
committed `data/published/davis-rates.svg` / `davis-ranked.md` always agree.

> The Davis dataset is **synthetic demonstration data**, not real reports.

In [ ]:
from pathlib import Path

from nearmiss.config import load_config
from nearmiss.engine import build_analysis
from nearmiss import figures

config = load_config(Path("..") / "config" / "davis-demo.toml")
bundle = build_analysis(config)
print("city:", config.city)
print("exposure coverage:", round(bundle.result.exposure_coverage * 100), "%")
print(
    "segments withheld (k-anonymity):", sum(1 for s in bundle.result.segments if not s.publishable)
)

## Ranked segments (exposure-normalized)

Rates are reports per unit of exposure, each with a 95% confidence interval and an `n`. The
`★` marks the statistically significant Getis-Ord Gi* cluster (FDR-corrected).

In [ ]:
from IPython.display import Markdown

Markdown(figures.render_ranked_md(config))

## Rate chart

The same data as a chart. The significant hotspot is drawn in a distinct color **and** a dashed
outline (a non-color pattern), with confidence-interval whiskers — risk is never conveyed by
color alone.

In [ ]:
from IPython.display import SVG

SVG(figures.render_bar_svg(config))

## Caveats

A high rate on a low-exposure segment still rests on few reports — read it with the interval.
Reporting bias is named, not hidden (see the brief and `docs/METHODOLOGY.md`). Regenerate the
committed figure and table with `make reproduce`.